In [1]:
import pandas as pd

In [2]:
import pandas as pd
import logging
from load_config import load_config

In [3]:
config = load_config()


In [7]:
df=pd.read_csv(path)

In [ ]:
df_num =df.select_dtypes(include ='number').drop(['transaction_id','is_fraud'],axis =1 )
df_cat =df.select_dtypes(exclude ='number')

X = df[df_num.columns.to_list() +df_cat.columns.to_list() ]
y = df['is_fraud']

In [10]:
from imblearn.pipeline import  Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, PowerTransformer
from sklearn.model_selection import train_test_split, cross_val_score,StratifiedKFold
from imblearn.over_sampling import SMOTE
from sklearn.feature_selection import RFECV
from lightgbm import LGBMClassifier
from sklearn.ensemble import RandomForestClassifier

X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.30, random_state=42, stratify=y)


In [11]:
tranform_num = Pipeline(steps=[('nums', PowerTransformer(method='yeo-johnson'))])
tranform_cat = Pipeline(steps = [('cat',OneHotEncoder(drop='first',sparse_output=False, handle_unknown='ignore'))])
tranform_cat_ordinal = Pipeline(steps = [('cat_ord', OrdinalEncoder())])

In [12]:
preprocess = ColumnTransformer(transformers=[('num',tranform_num,df_num.columns.to_list()),
																																													('cat',tranform_cat, df_cat[['merchant_category', 'card_type', 'auth_method', 'channel','device_type']].columns.to_list()),
                                             ('cat_ord',tranform_cat_ordinal , df_cat[['is_foreign_transaction', 'is_new_merchant', 'used_vpn','ip_country_mismatch', 'billing_shipping_mismatch','is_ai_generated_scam_attempt']].columns.to_list())                                                                                                                                       
																																												])

In [13]:
X_train_preprocess = preprocess.fit_transform(X_train)

from sklearn.ensemble import IsolationForest

iso = IsolationForest(contamination=0.01,random_state=42, n_estimators=1000, n_jobs=-1)

label = iso.fit_predict(X_train_preprocess)

mask= label ==1

X_train_clean = X_train[mask]
y_train_clean = y_train[mask]

In [14]:
model_LGBMClassifier = LGBMClassifier(
    n_estimators=500,
    learning_rate=0.05,
    num_leaves=31,
    max_depth=6,
    scale_pos_weight=99,      # Adjust this to match your exact fraud ratio
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)

In [15]:
select_RFECV = RFECV(estimator=model_LGBMClassifier, cv =10, min_features_to_select=20,n_jobs=-1, scoring='f1', verbose=True)

In [16]:
rf_model = RandomForestClassifier(
    n_estimators=200,
    criterion='gini',
    max_depth=15,                 # Caps depth to avoid memorizing individual fraud rows
    min_samples_split=10,         # Prevents splitting on tiny clusters of data
    min_samples_leaf=4,           # Ensures leaf nodes have generalized rules
    max_features='sqrt',
    class_weight='balanced',      # Crucial: Weights the fraud class higher
    random_state=42,
    n_jobs=-1                     # Uses all available CPU cores for speed
)

In [17]:
model = Pipeline(steps = [('preprocess',preprocess),
                          ('smote',SMOTE()),
                          ('select_RFECV',select_RFECV),
                          ('rf',rf_model)

																																																																																																								])

In [18]:
model.fit(X_train,y_train)

Fitting estimator with 49 features.
[LightGBM] [Info] Number of positive: 13763, number of negative: 13763
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.007677 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 12495
[LightGBM] [Info] Number of data points in the train set: 27526, number of used features: 49
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
Fitting estimator with 48 features.
[LightGBM] [Info] Number of positive: 13763, number of negative: 13763
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.008268 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 12240
[LightGBM] [Info] Number of data points in the train set: 27526, number of used features: 48
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Warning] No further splits with posi

,steps,"[('preprocess', ...), ('smote', ...), ...]"
,transform_input,None
,memory,None
,verbose,False
Name,Type,Value
classes_,"ndarray[int64](2,)","[0,1]"
feature_names_in_,"ndarray[object](24,)","['amount_usd','hours_since_last_txn','txn_count_last_24h',..., 'ip_country_mismatch','billing_shipping_mismatch', 'is_ai_generated_scam_attempt']"
n_features_in_,int,24
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3


In [21]:
y_pred= model.predict(X_test)

In [22]:
from sklearn.metrics import classification_report

In [23]:
print(classification_report(y_test,y_pred))

              precision    recall  f1-score   support

           0       0.98      1.00      0.99      5898
           1       0.57      0.04      0.07       102

    accuracy                           0.98      6000
   macro avg       0.78      0.52      0.53      6000
weighted avg       0.98      0.98      0.98      6000



In [28]:
import logging
from load_config import load_config
from load_csv import load_csv
from load_features import load_feature
from load_preprocess import load_preprocess
from load_model import load_model
from load_remove_outlier import remove_outlier
from load_ANN_model import create_model
from load_spilt_data import load_spilt_data

# Configure logging format to display clean timestamps and tracking levels
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

def main():
    try:
        logger.info("Initializing fraud detection pipeline execution...")
        
        # 1. Load data
        df = load_csv()
        logger.info(f"Step 1 Complete: Successfully loaded source CSV. Raw shape: {df.shape}")
        
        # 2. Extract features and targets
        df_num, df_cat, X, y = load_feature(df)
        logger.info(f"Step 2 Complete: Features extracted. Numeric columns: {df_num.shape}, Categorical columns: {df_cat.shape}")
        
        # 3. Create preprocessing transformations
        preprocess = load_preprocess(df_num, df_cat)
        logger.info("Step 3 Complete: Preprocessing configurations successfully created.")
        
        # 4. Initialize model pipeline setup
        model = load_model(preprocess)
        logger.info("Step 4 Complete: Model pipeline setup defined.")
        
        # 5. Stratified splitting of imbalanced dataset
        X_train_val, X_test_val, y_train_val, y_test_val = load_spilt_data(X, y)
        logger.info("Step 5 Complete: Data successfully split into training and test chunks.")
        logger.info(f" -> Training Data Size: {X_train_val.shape} | Test Data Size: {X_test_val.shape}")
        logger.info(f" -> Fraud Prevalence in Training Set: {y_train_val.mean():.4%}")
        
        # 6. Fit the model pipeline
        logger.info("Step 6: Launching model pipeline training (Preprocess -> Resample -> Train)...")
        model.fit(X_train_val, y_train_val)
        logger.info("Pipeline pipeline training completed successfully!")

								#7. Metrics Eval
        metrics = evaluate_fraud_metrics(model, X_test_val, y_test_val)			
        
        return None

    except Exception as e:
        logger.exception("A critical error halted the main pipeline execution loop.")

if __name__ == '__main__':
    main()


2026-09-19 11:05:56,186 - INFO - Initializing fraud detection pipeline execution...
2026-09-19 11:05:56,255 - INFO - Step 1 Complete: Successfully loaded source CSV. Raw shape: (20000, 26)
2026-09-19 11:05:56,259 - INFO - Step 2 Complete: Features extracted. Numeric columns: (20000, 13), Categorical columns: (20000, 11)
2026-09-19 11:05:56,261 - INFO - Step 3 Complete: Preprocessing configurations successfully created.
2026-09-19 11:05:56,264 - INFO - Step 4 Complete: Model pipeline setup defined.
2026-09-19 11:05:56,285 - INFO - Step 5 Complete: Data successfully split into training and test chunks.
2026-09-19 11:05:56,286 - INFO -  -> Training Data Size: (14000, 24) | Test Data Size: (6000, 24)
2026-09-19 11:05:56,288 - INFO -  -> Fraud Prevalence in Training Set: 1.6929%
2026-09-19 11:05:56,289 - INFO - Step 6: Launching model pipeline training (Preprocess -> Resample -> Train)...


Fitting estimator with 49 features.
[LightGBM] [Info] Number of positive: 13763, number of negative: 13763
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.010407 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 12495
[LightGBM] [Info] Number of data points in the train set: 27526, number of used features: 49
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
Fitting estimator with 48 features.
[LightGBM] [Info] Number of positive: 13763, number of negative: 13763
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.008958 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 12240
[LightGBM] [Info] Number of data points in the train set: 27526, number of used features: 48
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
Fitting estimator with 47 features.
[LightGBM] [

2026-09-19 11:14:09,262 - INFO - Pipeline pipeline training completed successfully!
2026-09-19 11:14:09,264 - INFO - Starting model evaluation on test data...
2026-09-19 11:14:09,498 - INFO - Predictions and probabilities generated successfully.
2026-09-19 11:14:09,502 - INFO - 
=== CONFUSION MATRIX ===
2026-09-19 11:14:09,503 - INFO - True Negatives (Legit caught): 5892
2026-09-19 11:14:09,505 - INFO - False Positives (False Alarms): 6
2026-09-19 11:14:09,506 - INFO - False Negatives (Missed Fraud!): 98
2026-09-19 11:14:09,507 - INFO - True Positives (Fraud caught!): 4
2026-09-19 11:14:09,529 - INFO - ROC-AUC Score: 0.8968
2026-09-19 11:14:09,530 - INFO - PR-AUC (Precision-Recall AUC) Score: 0.1782



=== CLASSIFICATION REPORT ===
              precision    recall  f1-score   support

  Legitimate       0.98      1.00      0.99      5898
       Fraud       0.40      0.04      0.07       102

    accuracy                           0.98      6000
   macro avg       0.69      0.52      0.53      6000
weighted avg       0.97      0.98      0.98      6000



In [26]:
import logging
from load_config import load_config
from load_csv import load_csv
from load_features import load_feature
from load_preprocess import load_preprocess
from load_model import load_model
from load_remove_outlier import remove_outlier
from load_ANN_model import create_model
from load_spilt_data import load_spilt_data
from load_metrics import evaluate_fraud_metrics  

# Configure logging format to display timestamp, level, and messages clearly
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

def main():
    try:
        logger.info("Step 1: Starting the main fraud detection execution pipeline...")
        
        # 1. Load data
        df = load_csv()
        logger.info(f"Successfully loaded CSV data. Shape: {df.shape}")
        
        # Optional Outlier Removal Step (Add if needed: df = remove_outlier(df))
        
        # 2. Extract features and targets
        df_num, df_cat, X, y = load_feature(df)
        logger.info(f"Features extracted. Numeric columns: {df_num.shape[1]}, Categorical columns: {df_cat.shape[1]}")
        
        # 3. Create preprocessing pipeline
        preprocess = load_preprocess(df_num, df_cat)
        logger.info("Preprocessing configurations initialized.")
        
        # 4. Stratified splitting of imbalanced data
        X_train_val, X_test_val, y_train_val, y_test_val = load_spilt_data(X, y)
        logger.info(f"Data split executed successfully.")
        logger.info(f"Training Set Size: {X_train_val.shape[0]} | Test Set Size: {X_test_val.shape[0]}")
        logger.info(f"Training Fraud Prevalence: {y_train_val.mean():.4%}")
        
        # 5. Build full Machine Learning pipeline (Preprocess -> SMOTE -> RFECV -> ANN)
        logger.info("Constructing the imblearn Pipeline components...")
        model = create_model(preprocess)
        
        # 6. Fit the model pipeline
        logger.info("Starting model training pipeline (Preprocessing, SMOTE Resampling, Feature Selection, ANN Fitting)...")
        model.fit(X_train_val, y_train_val)
        logger.info("Pipeline training completed successfully!")
        
        #7. Metrics Eval
        metrics = evaluate_fraud_metrics(model, X_test_val, y_test_val)
        return None
        
    except Exception as e:
        logger.exception('A critical error occurred in the main pipeline execution loop.')

if __name__ == '__main__':
    main()


2026-09-19 10:53:27,301 - INFO - Step 1: Starting the main fraud detection execution pipeline...
2026-09-19 10:53:27,363 - INFO - Successfully loaded CSV data. Shape: (20000, 26)
2026-09-19 10:53:27,366 - INFO - Features extracted. Numeric columns: 13, Categorical columns: 11
2026-09-19 10:53:27,367 - INFO - Preprocessing configurations initialized.
2026-09-19 10:53:27,385 - INFO - Data split executed successfully.
2026-09-19 10:53:27,386 - INFO - Training Set Size: 14000 | Test Set Size: 6000
2026-09-19 10:53:27,388 - INFO - Training Fraud Prevalence: 1.6929%
2026-09-19 10:53:27,389 - INFO - Constructing the imblearn Pipeline components...
2026-09-19 10:53:27,393 - INFO - Starting model training pipeline (Preprocessing, SMOTE Resampling, Feature Selection, ANN Fitting)...


[LightGBM] [Info] Number of positive: 13763, number of negative: 13763
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.008251 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 12495
[LightGBM] [Info] Number of data points in the train set: 27526, number of used features: 49
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Info] Number of positive: 13763, number of negative: 13763
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.006619 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 12495
[LightGBM] [Info] Number of data points in the train set: 27526, number of used features: 49
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Warning] No further splits with p

2026-09-19 10:59:13,693 - WARNING - TensorFlow GPU support is not available on native Windows for TensorFlow >= 2.11. Even if CUDA/cuDNN are installed, GPU will not be used. Please use WSL2 or the TensorFlow-DirectML plugin.


Epoch 1/50
1358/1377 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - auc: 0.9138 - loss: 0.3589 - precision: 0.7890 - recall: 0.7591
Epoch 1: val_recall improved from None to 0.90992, saving model to best_fraud_model.keras

Epoch 1: finished saving model to best_fraud_model.keras
1377/1377 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - auc: 0.9148 - loss: 0.3573 - precision: 0.7910 - recall: 0.7609 - val_auc: 0.0000e+00 - val_loss: 0.2563 - val_precision: 1.0000 - val_recall: 0.9099
Epoch 2/50
1373/1377 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - auc: 0.9713 - loss: 0.2045 - precision: 0.8765 - recall: 0.9092
Epoch 2: val_recall improved from 0.90992 to 0.95696, saving model to best_fraud_model.keras

Epoch 2: finished saving model to best_fraud_model.keras
1377/1377 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - auc: 0.9713 - loss: 0.2043 - precision: 0.8766 - recall: 0.9092 - val_auc: 0.0000e+00 - val_loss: 0.1708 - val_precision: 1.0000 - val_recall: 0.9570
Epoch 3/50
1366/1377 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - auc: 0.9812 - l

2026-09-19 11:02:09,544 - INFO - Pipeline training completed successfully!
2026-09-19 11:02:09,545 - INFO - Starting model evaluation on test data...


375/375 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step
375/375 ━━━━━━━━━━━━━━━━━━━━ 0s 986us/step


2026-09-19 11:02:10,723 - INFO - Predictions and probabilities generated successfully.
2026-09-19 11:02:10,728 - INFO - 
=== CONFUSION MATRIX ===
2026-09-19 11:02:10,730 - INFO - True Negatives (Legit caught): 5855
2026-09-19 11:02:10,730 - INFO - False Positives (False Alarms): 43
2026-09-19 11:02:10,731 - INFO - False Negatives (Missed Fraud!): 86
2026-09-19 11:02:10,731 - INFO - True Positives (Fraud caught!): 16
2026-09-19 11:02:10,757 - INFO - ROC-AUC Score: 0.8520



=== CLASSIFICATION REPORT ===
              precision    recall  f1-score   support

  Legitimate       0.99      0.99      0.99      5898
       Fraud       0.27      0.16      0.20       102

    accuracy                           0.98      6000
   macro avg       0.63      0.57      0.59      6000
weighted avg       0.97      0.98      0.98      6000



2026-09-19 11:02:10,758 - INFO - PR-AUC (Precision-Recall AUC) Score: 0.1537
